# HateXplain preprocessing

Run all code cells in order using Python 3.9 or later. Install dependencies into the active Jupyter kernel using the included `%pip` cells.

## Paths and reproducibility

Run Jupyter from your repository root, or from its `notebooks/` directory. Alternatively, set `PROJECT_ROOT` in the download cell. All data paths are relative to that root; no personal desktop path is required.

Official source: https://github.com/hate-alert/HateXplain

Pinned revision: `01d742279dac941981f53806154481c0e15ee686`. The two official files are preserved byte-for-byte and checked against SHA-256 values embedded in the code. No download manifest is created.

## Cleaning policy

1. Include only official train, validation and test IDs; exclude unassigned samples.
2. Join official tokens with one space and assign strict-majority labels; map `hatespeech` to `hate`.
3. Exclude every record in an exact-text group with conflicting final labels.
4. Keep one record per remaining text, prioritizing test, validation, then train; break ties by sample ID.
5. Export only `sample_id`, `text`, `label`, and `split`. Do not modify spelling, placeholders, case, punctuation or emoji.

This produces a derived split, not the unchanged official benchmark. Exact deduplication does not detect all near duplicates.

## Outputs

- `data/HateXplain/raw/dataset.json`
- `data/HateXplain/raw/post_id_divisions.json`
- `data/HateXplain/clean/hatexplain.jsonl`
- `data/HateXplain/audit/cleaning_log.csv`

Identical saved results can be reused on reruns. Different existing outputs are never silently overwritten.

## Preserved execution output

All original code-cell outputs below are retained unchanged, including historical local paths and package-install logs. They document the original run, not execution of this revised notebook. On rerun, path messages and installation logs will reflect your environment. Cleaning rules, sample ordering and displayed statistical content are unchanged for the pinned data. The original run retained 19,186 samples (24 conflict removals and 19 duplicate removals).


In [1]:
%pip install requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import json
import requests

# Run from the repository root or its notebooks directory.
# Set PROJECT_ROOT explicitly if your Jupyter working directory is elsewhere.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / "data" / "HateXplain" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

REVISION = "01d742279dac941981f53806154481c0e15ee686"
EXPECTED_HASHES = {'dataset.json': '63bb3340fee0ec469b09690d04cb68f7c187787dd8b83807f071892c084967fb', 'post_id_divisions.json': 'c2fb0d89862e7897b11ea3e9380753f15a793482b4b70ad0532dfb1212212835'}

import hashlib
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=3, backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
)))

def validate_payload(name, content):
    json.loads(content)
    actual = hashlib.sha256(content).hexdigest()
    if actual != EXPECTED_HASHES[name]:
        raise ValueError(f"Unexpected content for {name}; expected revision {REVISION}.")

def reject_duplicate_keys(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"Duplicate JSON key: {key}")
        result[key] = value
    return result

file_names = ["dataset.json", "post_id_divisions.json"]
existing = [(RAW_DIR / name).exists() for name in file_names]

if all(existing):
    print("Both official files already exist. Loading without overwriting.")

elif any(existing):
    raise RuntimeError(
        "Only one file exists. Supply both files from the same revision "
        "or move the existing file before retrying."
    )

else:
    revision = REVISION

    print("Pinned revision:", revision)

    downloaded = {}

    for name in file_names:
        url = (
            "https://raw.githubusercontent.com/"
            f"hate-alert/HateXplain/{revision}/Data/{name}"
        )

        response = session.get(url, timeout=120)
        response.raise_for_status()

        validate_payload(name, response.content)
        downloaded[name] = response.content

        print(f"{name} download URL: {url}")

    # Save only after both downloads have succeeded.
    for name, content in downloaded.items():
        (RAW_DIR / name).write_bytes(content)

    print("\nDownload complete.")

# Validate local files too, including files from earlier runs.
for name in file_names:
    validate_payload(name, (RAW_DIR / name).read_bytes())

dataset = json.loads(
    (RAW_DIR / "dataset.json").read_text(encoding="utf-8"),
    object_pairs_hook=reject_duplicate_keys
)

official_divisions = json.loads(
    (RAW_DIR / "post_id_divisions.json").read_text(encoding="utf-8"),
    object_pairs_hook=reject_duplicate_keys
)

print("\nSave location:", RAW_DIR)
print("Total original samples:", len(dataset))

for split, sample_ids in official_divisions.items():
    print(f"{split}: {len(sample_ids):,} samples")

Pinned revision: 01d742279dac941981f53806154481c0e15ee686
dataset.json download URL: https://raw.githubusercontent.com/hate-alert/HateXplain/01d742279dac941981f53806154481c0e15ee686/Data/dataset.json
post_id_divisions.json download URL: https://raw.githubusercontent.com/hate-alert/HateXplain/01d742279dac941981f53806154481c0e15ee686/Data/post_id_divisions.json

Download complete.

Save location: /Users/felicia/Desktop/项目一/data/HateXplain/raw
Total original samples: 20148
test: 1,924 samples
train: 15,383 samples
val: 1,922 samples


In [3]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [4]:
from collections import Counter
import pandas as pd
from IPython.display import display

split_mapping = {
    "train": "train",
    "val": "validation",
    "valid": "validation",
    "validation": "validation",
    "test": "test",
}

label_mapping = {
    "hatespeech": "hate",
    "offensive": "offensive",
    "normal": "normal",
}

rows = []
seen_ids = set()
seen_splits = set()

for original_split, sample_ids in official_divisions.items():
    if original_split not in split_mapping:
        raise ValueError(f"Unexpected split: {original_split}")

    split = split_mapping[original_split]

    if split in seen_splits:
        raise ValueError(f"Duplicate split definition: {split}")

    seen_splits.add(split)

    if not isinstance(sample_ids, list):
        raise TypeError(f"Invalid ID list: {original_split}")

    for sample_id in sample_ids:
        if sample_id in seen_ids:
            raise ValueError(f"Repeated sample ID in splits: {sample_id}")

        if sample_id not in dataset:
            raise ValueError(f"Sample not found: {sample_id}")

        seen_ids.add(sample_id)
        sample = dataset[sample_id]
        if sample.get("post_id") != sample_id:
            raise ValueError(f"Inconsistent post_id: {sample_id}")

        tokens = sample.get("post_tokens")

        if not isinstance(tokens, list) or not all(
            isinstance(token, str) for token in tokens
        ):
            raise ValueError(f"Invalid tokens: {sample_id}")

        votes = [
            annotator.get("label")
            for annotator in sample.get("annotators", [])
        ]

        if not votes or any(vote not in label_mapping for vote in votes):
            raise ValueError(f"Missing or invalid labels: {sample_id}")

        majority_label, vote_count = Counter(votes).most_common(1)[0]

        if vote_count <= len(votes) / 2:
            raise ValueError(f"No strict majority label: {sample_id}")

        rows.append({
            "sample_id": sample_id,
            "text": " ".join(tokens),
            "label": label_mapping[majority_label],
            "split": split,
        })

if seen_splits != {"train", "validation", "test"}:
    raise ValueError("One or more official splits are missing.")

# Only officially assigned samples are included.
df = pd.DataFrame(
    rows,
    columns=["sample_id", "text", "label", "split"],
)

unassigned_count = len(set(dataset) - seen_ids)

print("Original samples:", len(dataset))
print("Excluded unassigned samples:", unassigned_count)
print("Included official samples:", len(df))

print("\nSample counts by split:")
display(
    df.groupby("split")
    .size()
    .reindex(["train", "validation", "test"])
    .rename("sample_count")
    .to_frame()
)

print("\nLabel distribution:")
display(
    pd.crosstab(df["split"], df["label"]).reindex(
        index=["train", "validation", "test"],
        columns=["hate", "offensive", "normal"],
        fill_value=0,
    )
)

print("\nFirst five samples:")
display(df.head())

Original samples: 20148
Excluded unassigned samples: 919
Included official samples: 19229

Sample counts by split:


,sample_count
split,
train,15383
validation,1922
test,1924



Label distribution:


label,hate,offensive,normal
split,,,
train,4748,4384,6251
validation,593,548,781
test,594,548,782



First five samples:


,sample_id,text,label,split
0,1178610029273976833_twitter,<user> men can not be raped can not be abused ...,normal,test
1,1165785686903009283_twitter,<user> you are missing an essential premise wh...,normal,test
2,1252707503441313794_twitter,<user> <user> why are you repeating yourself a...,offensive,test
3,1103385226921762816_twitter,<user> <user> well she ’ muslim so of course s...,hate,test
4,1169443635869487105_twitter,<user> lol not me i don ’ t deal with bms not ...,offensive,test


In [5]:
# Find exact duplicate texts within the official splits.
duplicate_rows = df.loc[
    df["text"].duplicated(keep=False)
].copy()

group_records = []
detail_parts = []

split_priority = {"train": 0, "validation": 1, "test": 2}

for number, (text, group) in enumerate(
    duplicate_rows.groupby("text", sort=True),
    start=1,
):
    group_id = f"D{number:03d}"

    splits = sorted(
        group["split"].unique(),
        key=lambda name: split_priority[name],
    )

    labels = sorted(group["label"].unique())

    group_records.append({
        "group_id": group_id,
        "sample_count": len(group),
        "splits": " / ".join(splits),
        "labels": " / ".join(labels),
        "cross_split": len(splits) > 1,
        "label_conflict": len(labels) > 1,
    })

    detail = group[
        ["sample_id", "split", "label", "text"]
    ].copy()

    detail.insert(0, "group_id", group_id)
    detail_parts.append(detail)

duplicate_groups = pd.DataFrame(
    group_records,
    columns=[
        "group_id", "sample_count", "splits",
        "labels", "cross_split", "label_conflict",
    ],
)

duplicate_details = (
    pd.concat(detail_parts, ignore_index=True)
    if detail_parts
    else pd.DataFrame(
        columns=["group_id", "sample_id", "split", "label", "text"]
    )
)

cross_groups = duplicate_groups.loc[
    duplicate_groups["cross_split"].eq(True)
]

conflict_groups = duplicate_groups.loc[
    duplicate_groups["label_conflict"].eq(True)
]

print("Duplicate groups:", len(duplicate_groups))
print("Samples in duplicate groups:", len(duplicate_details))
print("Cross-split duplicate groups:", len(cross_groups))
print("Conflicting-label groups:", len(conflict_groups))

print("\nAll duplicate groups:")
display(duplicate_groups)

print("\nCross-split duplicate groups:")
display(cross_groups)

print("\nConflicting-label groups:")
display(conflict_groups)

print("\nDuplicate sample details:")

with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", None,
):
    display(
        duplicate_details.sort_values(
            ["group_id", "split", "sample_id"]
        )
    )

Duplicate groups: 22
Samples in duplicate groups: 59
Cross-split duplicate groups: 7
Conflicting-label groups: 6

All duplicate groups:


,group_id,sample_count,splits,labels,cross_split,label_conflict
0,D001,3,train,normal,False,False
1,D002,2,train / test,offensive,True,False
2,D003,2,train / validation,offensive,True,False
3,D004,2,train,hate / offensive,False,True
4,D005,3,train / test,normal / offensive,True,True
5,D006,2,train,normal,False,False
6,D007,2,train / test,normal,True,False
7,D008,13,train,hate / normal / offensive,False,True
8,D009,2,train,normal / offensive,False,True
9,D010,2,train,normal,False,False



Cross-split duplicate groups:


,group_id,sample_count,splits,labels,cross_split,label_conflict
1,D002,2,train / test,offensive,True,False
2,D003,2,train / validation,offensive,True,False
4,D005,3,train / test,normal / offensive,True,True
6,D007,2,train / test,normal,True,False
11,D012,4,train / validation / test,normal,True,False
12,D013,2,train / validation,offensive,True,False
20,D021,2,train / test,normal / offensive,True,True



Conflicting-label groups:


,group_id,sample_count,splits,labels,cross_split,label_conflict
3,D004,2,train,hate / offensive,False,True
4,D005,3,train / test,normal / offensive,True,True
7,D008,13,train,hate / normal / offensive,False,True
8,D009,2,train,normal / offensive,False,True
15,D016,2,train,normal / offensive,False,True
20,D021,2,train / test,normal / offensive,True,True



Duplicate sample details:


,group_id,sample_id,split,label,text
2,D001,1264198113834618881_twitter,train,normal,<user> <number> bank islam
0,D001,1264228895827480576_twitter,train,normal,<user> <number> bank islam
1,D001,1264509142451941376_twitter,train,normal,<user> <number> bank islam
3,D002,1107036788005179392_twitter,test,offensive,<user> <user> are you retarded
4,D002,1184411429132263424_twitter,train,offensive,<user> <user> are you retarded
5,D003,1172986514147381249_twitter,train,offensive,<user> are you a fucking retard
6,D003,1171620659996319744_twitter,validation,offensive,<user> are you a fucking retard
7,D004,1178330971004751872_twitter,train,offensive,<user> ching chong
8,D004,1178963159907852288_twitter,train,hate,<user> ching chong
9,D005,1178818457112104962_twitter,test,normal,<user> i fucking hate you


In [6]:
# Remove all samples belonging to conflicting-label text groups.
label_counts = df.groupby("text")["label"].nunique()
conflicting_texts = label_counts[label_counts > 1].index

conflict_mask = df["text"].isin(conflicting_texts)

conflict_removed = df.loc[conflict_mask].copy()
remaining = df.loc[~conflict_mask].copy()

# Keep one sample per text: test > validation > train.
split_rank = {"test": 0, "validation": 1, "train": 2}

remaining["_split_rank"] = remaining["split"].map(split_rank)

remaining = remaining.sort_values(
    ["_split_rank", "sample_id"],
    kind="stable",
)

duplicate_mask = remaining["text"].duplicated(keep="first")

duplicate_removed = remaining.loc[duplicate_mask].copy()

clean_df = remaining.loc[
    ~duplicate_mask,
    ["sample_id", "text", "label", "split"],
].copy()

# Record which sample was retained for each duplicate text.
retained_ids = clean_df.set_index("text")["sample_id"]

conflict_removed["removal_reason"] = "conflicting_labels"
conflict_removed["retained_sample_id"] = None

duplicate_removed["removal_reason"] = "duplicate_text"
duplicate_removed["retained_sample_id"] = (
    duplicate_removed["text"].map(retained_ids)
)

log_columns = [
    "sample_id",
    "text",
    "label",
    "split",
    "removal_reason",
    "retained_sample_id",
]

cleaning_log = pd.concat(
    [
        conflict_removed[log_columns],
        duplicate_removed[log_columns],
    ],
    ignore_index=True,
)

clean_df = clean_df.sort_values(
    ["split", "sample_id"]
).reset_index(drop=True)

# Verify the cleaned data.
assert clean_df["sample_id"].is_unique
assert clean_df["text"].is_unique
assert not clean_df.isna().any().any()
assert clean_df["text"].str.strip().ne("").all()
assert clean_df["split"].isin(["train", "validation", "test"]).all()
assert clean_df["label"].isin(["hate", "offensive", "normal"]).all()
assert len(df) == len(clean_df) + len(cleaning_log)

# Audit completeness and retained-ID integrity.
assert cleaning_log["sample_id"].is_unique
assert set(clean_df["sample_id"]).isdisjoint(cleaning_log["sample_id"])
assert set(clean_df["sample_id"]) | set(cleaning_log["sample_id"]) == set(df["sample_id"])
assert not clean_df["text"].isin(conflicting_texts).any()
assert duplicate_removed["retained_sample_id"].isin(clean_df["sample_id"]).all()

print("Samples before cleaning:", len(df))
print("Removed due to conflicting labels:", len(conflict_removed))
print("Removed as duplicate texts:", len(duplicate_removed))
print("Samples after cleaning:", len(clean_df))

print("\nSample counts before and after cleaning:")

count_summary = pd.DataFrame({
    "before": df.groupby("split").size(),
    "after": clean_df.groupby("split").size(),
}).reindex(
    ["train", "validation", "test"]
).fillna(0).astype(int)

count_summary["removed"] = (
    count_summary["before"] - count_summary["after"]
)

display(count_summary)

print("\nCleaned label distribution:")

display(
    pd.crosstab(clean_df["split"], clean_df["label"]).reindex(
        index=["train", "validation", "test"],
        columns=["hate", "offensive", "normal"],
        fill_value=0,
    )
)

print("\nRemoval log:")
display(cleaning_log)

print("\nFirst five cleaned samples:")
display(clean_df.head())

Samples before cleaning: 19229
Removed due to conflicting labels: 24
Removed as duplicate texts: 19
Samples after cleaning: 19186

Sample counts before and after cleaning:


,before,after,removed
split,,,
train,15383,15344,39
validation,1922,1921,1
test,1924,1921,3



Cleaned label distribution:


label,hate,offensive,normal
split,,,
train,4743,4372,6229
validation,593,548,780
test,594,548,779



Removal log:


,sample_id,text,label,split,removal_reason,retained_sample_id
0,6710056_gab,teacher asks class where is pakistan little jo...,normal,test,conflicting_labels,None
1,1178818457112104962_twitter,<user> i fucking hate you,normal,test,conflicting_labels,None
2,1178861841780559872_twitter,<user> i fucking hate you,normal,test,conflicting_labels,None
3,1193293522142547968_twitter,<user> i hate you,normal,train,conflicting_labels,None
4,1278613802464931841_twitter,<user> i hate you,hate,train,conflicting_labels,None
5,1089320840620134400_twitter,<user> i hate you,normal,train,conflicting_labels,None
6,1204096845514981384_twitter,<user> i hate you,normal,train,conflicting_labels,None
7,1092620747015680000_twitter,<user> i hate you,hate,train,conflicting_labels,None
8,1257365940737585152_twitter,<user> i hate you,offensive,train,conflicting_labels,None
9,1125772131588087809_twitter,<user> i hate you 😂,offensive,train,conflicting_labels,None



First five cleaned samples:


,sample_id,text,label,split
0,10038094_gab,i do i do i love retarded conspiracies lol so ...,offensive,test
1,10040607_gab,recordbreakerr recordbreakerrrrrrrrrrrrrr5 hou...,hate,test
2,10053426_gab,ventura only governor to refund overpaid taxes...,offensive,test
3,10068208_gab,<user> i went to a minor league ball game yest...,normal,test
4,10088571_gab,jews are the only enemy in the history of the ...,hate,test


In [7]:
CLEAN_DIR = RAW_DIR.parent / "clean"
AUDIT_DIR = RAW_DIR.parent / "audit"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

clean_path = CLEAN_DIR / "hatexplain.jsonl"
log_path = AUDIT_DIR / "cleaning_log.csv"

# Serialize first. Existing identical results are safe to reuse.
# Different results require explicit removal or a different output directory.
clean_content = clean_df.to_json(
    orient="records", lines=True, force_ascii=False,
).encode("utf-8")
log_content = cleaning_log.to_csv(index=False).encode("utf-8-sig")

outputs_to_save = [(clean_path, clean_content), (log_path, log_content)]
for path, content in outputs_to_save:
    if path.exists() and path.read_bytes() != content:
        raise FileExistsError(f"Existing output differs; move it before retrying: {path}")

for path, content in outputs_to_save:
    if not path.exists():
        temporary_path = path.with_suffix(path.suffix + ".tmp")
        temporary_path.write_bytes(content)
        temporary_path.replace(path)

# Reload and verify the saved dataset.
saved_df = pd.read_json(
    clean_path,
    lines=True,
    dtype={
        "sample_id": "string",
        "text": "string",
        "label": "string",
        "split": "string",
    },
    convert_dates=False,
)

pd.testing.assert_frame_equal(
    saved_df,
    clean_df.astype("string"),
    check_dtype=False,
)

print("Saved dataset:", clean_path)
print("Saved cleaning log:", log_path)
print("Verified samples:", len(saved_df))
print("Fields:", saved_df.columns.tolist())

Saved dataset: /Users/felicia/Desktop/项目一/data/HateXplain/clean/hatexplain.jsonl
Saved cleaning log: /Users/felicia/Desktop/项目一/data/HateXplain/audit/cleaning_log.csv
Verified samples: 19186
Fields: ['sample_id', 'text', 'label', 'split']
